# JSBSim C172 Empirical Level-Flight Condition Refinement v49

This notebook refines the promising region found by v48:

- aircraft: `c172p`
- speed: near `100 kt`
- pitch: near `-4 deg`
- elevator: near `0.15`
- throttle: near `0.80`

The goal is to find a practical, repeatable initial condition for controller tests, even if exact JSBSim trim is unavailable.

Compared with v48, this version adds:

1. a local fine grid around the best coarse region,
2. relaxed and strict feasibility labels,
3. early-window-excluded metrics, because the first few seconds can contain initialization/actuator settling,
4. a paper-friendly selected initial-condition table.

In [1]:
try:
    import jsbsim
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'jsbsim', '-q'])
    import jsbsim
print('jsbsim import ok')

jsbsim import ok


In [2]:
import os, time, math, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jsbsim

try:
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB_DRIVE_MOUNTED = True
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)
    COLAB_DRIVE_MOUNTED = False

FPS_PER_KT = 1.687809857
EXPERIMENT_NAME = 'jsbsim_c172_empirical_level_flight_refine_v49'
RUN_MODE = 'local_refine'
RESULT_ROOT = '/content/drive/MyDrive/Colab Result' if COLAB_DRIVE_MOUNTED else './Colab Result'
SAVE_DIR = os.path.join(RESULT_ROOT, 'PINN_MPC', EXPERIMENT_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)
print('SAVE_DIR:', SAVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SAVE_DIR: /content/drive/MyDrive/Colab Result/PINN_MPC/jsbsim_c172_empirical_level_flight_refine_v49


## 1. Refined Local Grid

In [3]:
AIRCRAFT = 'c172p'
INIT_ALT_FT = 3000.0
MIXTURE = 1.0

# Refine around the best v48 region.
SPEED_GRID_KTS = np.arange(96.0, 105.01, 1.0)
PITCH_GRID_DEG = np.arange(-5.0, -2.0 + 1e-9, 0.25)
ELEVATOR_GRID = np.arange(0.120, 0.180 + 1e-9, 0.005)
THROTTLE_GRID = np.arange(0.700, 0.830 + 1e-9, 0.010)

DT = 0.02
SETTLE_SECONDS = 2.0
HOLD_SECONDS = 60.0
TAIL_SECONDS = 15.0
METRIC_SKIP_SECONDS = 5.0
TOP_K = 40

ALT_ABORT_DELTA_FT = 800.0
PITCH_ABORT_DEG = 25.0
SPEED_ABORT_KTS = 45.0

print('refine grid size:', len(SPEED_GRID_KTS) * len(PITCH_GRID_DEG) * len(ELEVATOR_GRID) * len(THROTTLE_GRID))

refine grid size: 23660


In [4]:
def safe_set(fdm, prop, value):
    try:
        fdm[prop] = value
        return True
    except Exception:
        return False


def get_prop(fdm, prop, default=np.nan):
    try:
        return float(fdm[prop])
    except Exception:
        return float(default)


def read_state(fdm):
    return {
        'time_s': get_prop(fdm, 'simulation/sim-time-sec'),
        'h_sl_ft': get_prop(fdm, 'position/h-sl-ft'),
        'vc_kts': get_prop(fdm, 'velocities/vc-kts'),
        'vt_fps': get_prop(fdm, 'velocities/vt-fps'),
        'theta_deg': math.degrees(get_prop(fdm, 'attitude/theta-rad', 0.0)),
        'alpha_deg': math.degrees(get_prop(fdm, 'aero/alpha-rad', 0.0)),
        'q_deg_s': math.degrees(get_prop(fdm, 'velocities/q-rad_sec', 0.0)),
        'qdot_deg_s2': math.degrees(get_prop(fdm, 'accelerations/qdot-rad_sec2', 0.0)),
        'elevator_cmd': get_prop(fdm, 'fcs/elevator-cmd-norm'),
        'elevator_pos': get_prop(fdm, 'fcs/elevator-pos-norm'),
        'throttle_cmd': get_prop(fdm, 'fcs/throttle-cmd-norm'),
        'throttle_pos': get_prop(fdm, 'fcs/throttle-pos-norm[0]'),
        'rpm': get_prop(fdm, 'propulsion/engine/propeller-rpm'),
    }


def make_fdm_fixed_ic(speed_kts, pitch_deg, elevator, throttle):
    fdm = jsbsim.FGFDMExec(None, None)
    fdm.set_debug_level(0)
    fdm.load_model(AIRCRAFT)

    for prop in ['ic/h-sl-ft', 'ic/altitude-ft']:
        safe_set(fdm, prop, INIT_ALT_FT)
    for prop in ['ic/vc-kts', 'ic/vt-kts']:
        safe_set(fdm, prop, speed_kts)
    safe_set(fdm, 'ic/u-fps', speed_kts * FPS_PER_KT)
    safe_set(fdm, 'ic/ubody-fps', speed_kts * FPS_PER_KT)
    safe_set(fdm, 'ic/v-fps', 0.0)
    safe_set(fdm, 'ic/w-fps', 0.0)
    safe_set(fdm, 'ic/p-rad_sec', 0.0)
    safe_set(fdm, 'ic/q-rad_sec', 0.0)
    safe_set(fdm, 'ic/r-rad_sec', 0.0)
    safe_set(fdm, 'ic/phi-deg', 0.0)
    safe_set(fdm, 'ic/theta-deg', pitch_deg)
    safe_set(fdm, 'ic/psi-true-deg', 200.0)
    safe_set(fdm, 'ic/lat-gc-deg', 47.0)
    safe_set(fdm, 'ic/long-gc-deg', -122.0)
    fdm.run_ic()

    for prop, val in [
        ('fcs/elevator-cmd-norm', elevator),
        ('fcs/elevator-pos-norm', elevator),
        ('fcs/throttle-cmd-norm', throttle),
        ('fcs/throttle-pos-norm[0]', throttle),
        ('fcs/mixture-cmd-norm', MIXTURE),
        ('fcs/mixture-pos-norm[0]', MIXTURE),
        ('propulsion/magneto_cmd', 3),
        ('propulsion/starter_cmd', 1),
        ('fcs/flap-cmd-norm', 0.0),
        ('fcs/flap-pos-deg', 0.0),
        ('gear/gear-cmd-norm', 1.0),
        ('gear/gear-pos-norm', 1.0),
    ]:
        safe_set(fdm, prop, val)
    return fdm


def window_metrics(df, prefix, init_alt_ft=INIT_ALT_FT):
    if len(df) < 5:
        return {f'{prefix}_valid': False}
    t = df['time_s'].to_numpy(dtype=float)
    h = df['h_sl_ft'].to_numpy(dtype=float)
    v = df['vc_kts'].to_numpy(dtype=float)
    theta = df['theta_deg'].to_numpy(dtype=float)
    q = df['q_deg_s'].to_numpy(dtype=float)
    alpha = df['alpha_deg'].to_numpy(dtype=float)
    tail_n = max(3, min(len(df), int(TAIL_SECONDS / DT)))
    tail = df.tail(tail_n)
    tail_dt = max(float(tail['time_s'].iloc[-1] - tail['time_s'].iloc[0]), 1e-6)
    return {
        f'{prefix}_valid': True,
        f'{prefix}_duration_s': float(max(t[-1] - t[0], 0.0)),
        f'{prefix}_alt_drift_ft': float(h[-1] - h[0]),
        f'{prefix}_alt_rmse_ft': float(np.sqrt(np.mean((h - init_alt_ft) ** 2))),
        f'{prefix}_tail_vs_fps': float((tail['h_sl_ft'].iloc[-1] - tail['h_sl_ft'].iloc[0]) / tail_dt),
        f'{prefix}_speed_drift_kts': float(v[-1] - v[0]),
        f'{prefix}_theta_drift_deg': float(theta[-1] - theta[0]),
        f'{prefix}_q_rms_deg_s': float(np.sqrt(np.mean(q ** 2))),
        f'{prefix}_tail_q_rms_deg_s': float(np.sqrt(np.mean(tail['q_deg_s'].to_numpy(dtype=float) ** 2))),
        f'{prefix}_max_abs_pitch_deg': float(np.max(np.abs(theta))),
        f'{prefix}_max_abs_alpha_deg': float(np.max(np.abs(alpha))),
        f'{prefix}_final_alt_ft': float(h[-1]),
        f'{prefix}_final_speed_kts': float(v[-1]),
        f'{prefix}_final_pitch_deg': float(theta[-1]),
    }


def classify_feasibility(m):
    relaxed = (
        abs(m['eval_alt_drift_ft']) < 25.0
        and abs(m['eval_tail_vs_fps']) < 0.75
        and abs(m['eval_speed_drift_kts']) < 6.0
        and m['eval_q_rms_deg_s'] < 3.0
        and m['eval_tail_q_rms_deg_s'] < 4.0
        and m['eval_max_abs_pitch_deg'] < 10.0
        and m['eval_max_abs_alpha_deg'] < 8.0
    )
    strict = (
        abs(m['eval_alt_drift_ft']) < 10.0
        and abs(m['eval_tail_vs_fps']) < 0.35
        and abs(m['eval_speed_drift_kts']) < 3.0
        and m['eval_q_rms_deg_s'] < 1.0
        and m['eval_tail_q_rms_deg_s'] < 1.5
        and m['eval_max_abs_pitch_deg'] < 8.0
        and m['eval_max_abs_alpha_deg'] < 6.0
    )
    return bool(relaxed), bool(strict)


def score_metrics(m):
    # Main score uses post-settle/evaluation window. Lower is better.
    return float(
        1.00 * abs(m['eval_alt_drift_ft'])
        + 20.0 * abs(m['eval_tail_vs_fps'])
        + 4.00 * abs(m['eval_speed_drift_kts'])
        + 10.0 * m['eval_q_rms_deg_s']
        + 8.00 * m['eval_tail_q_rms_deg_s']
        + 5.00 * abs(m['eval_theta_drift_deg'])
        + 0.20 * m['eval_alt_rmse_ft']
        + 0.25 * m['eval_max_abs_alpha_deg']
    )


def run_case(speed_kts, pitch_deg, elevator, throttle, store_log=False):
    row = {
        'aircraft': AIRCRAFT,
        'init_alt_ft': INIT_ALT_FT,
        'init_speed_kts': float(speed_kts),
        'init_pitch_deg': float(pitch_deg),
        'elevator': float(elevator),
        'throttle': float(throttle),
        'mixture': MIXTURE,
        'valid': False,
        'abort': 'None',
        'score': np.inf,
    }
    rows = []
    try:
        fdm = make_fdm_fixed_ic(speed_kts, pitch_deg, elevator, throttle)
        for _ in range(int(SETTLE_SECONDS / DT)):
            safe_set(fdm, 'fcs/elevator-cmd-norm', elevator)
            safe_set(fdm, 'fcs/throttle-cmd-norm', throttle)
            fdm.run()
        h0 = read_state(fdm)['h_sl_ft']
        for _ in range(int(HOLD_SECONDS / DT)):
            safe_set(fdm, 'fcs/elevator-cmd-norm', elevator)
            safe_set(fdm, 'fcs/throttle-cmd-norm', throttle)
            fdm.run()
            s = read_state(fdm)
            rows.append(s)
            if abs(s['h_sl_ft'] - h0) > ALT_ABORT_DELTA_FT:
                row['abort'] = 'ALT_ABORT'
                break
            if abs(s['theta_deg']) > PITCH_ABORT_DEG:
                row['abort'] = 'PITCH_ABORT'
                break
            if s['vc_kts'] < SPEED_ABORT_KTS:
                row['abort'] = 'SPEED_ABORT'
                break
        df = pd.DataFrame(rows)
        if len(df) < 5:
            return row, df if store_log else None
        eval_df = df[df['time_s'] >= df['time_s'].iloc[0] + METRIC_SKIP_SECONDS].copy()
        if len(eval_df) < 5:
            eval_df = df.copy()
        row.update(window_metrics(df, 'full'))
        row.update(window_metrics(eval_df, 'eval'))
        row['valid'] = bool(row.get('eval_valid', False))
        row['feasible_relaxed'], row['feasible_strict'] = classify_feasibility(row)
        row['score'] = score_metrics(row)
        return row, df if store_log else None
    except Exception as exc:
        row['abort'] = f'exception:{type(exc).__name__}'
        row['exception_message'] = str(exc)
        return row, None

## 2. Run Local Refinement

In [5]:
rows = []
best_key = None
best_score = np.inf
total = len(SPEED_GRID_KTS) * len(PITCH_GRID_DEG) * len(ELEVATOR_GRID) * len(THROTTLE_GRID)
print('local refine cases:', total)
t0 = time.time()
idx = 0
for speed_kts in SPEED_GRID_KTS:
    for pitch_deg in PITCH_GRID_DEG:
        for elevator in ELEVATOR_GRID:
            for throttle in THROTTLE_GRID:
                idx += 1
                row, _ = run_case(speed_kts, pitch_deg, elevator, throttle, store_log=False)
                rows.append(row)
                if np.isfinite(row['score']) and row['score'] < best_score:
                    best_score = row['score']
                    best_key = (speed_kts, pitch_deg, elevator, throttle)
                if idx % 500 == 0 or idx == total:
                    tmp = pd.DataFrame(rows).sort_values('score')
                    b = tmp.iloc[0]
                    relaxed_n = int(tmp.get('feasible_relaxed', pd.Series(dtype=bool)).fillna(False).sum())
                    strict_n = int(tmp.get('feasible_strict', pd.Series(dtype=bool)).fillna(False).sum())
                    print(f'{idx}/{total} elapsed={time.time()-t0:.1f}s relaxed={relaxed_n} strict={strict_n} best score={b.score:.2f} V={b.init_speed_kts:.1f} pitch={b.init_pitch_deg:.2f} elev={b.elevator:.3f} thr={b.throttle:.2f}')

refine_df = pd.DataFrame(rows).sort_values('score').reset_index(drop=True)
refine_path = os.path.join(SAVE_DIR, f'empirical_level_hold_refine_v49_{RUN_MODE}.csv')
refine_df.to_csv(refine_path, index=False)
print('Saved:', refine_path)
print('Top refined candidates:')
display(refine_df.head(TOP_K))
print('Relaxed feasible count:', int(refine_df['feasible_relaxed'].fillna(False).sum()))
print('Strict feasible count:', int(refine_df['feasible_strict'].fillna(False).sum()))

local refine cases: 23660


     JSBSim Flight Dynamics Model v1.3.0 Apr  9 2026 10:00:08
            [JSBSim-ML v2.0]

JSBSim startup beginning ...

500/23660 elapsed=54.1s relaxed=0 strict=0 best score=340.99 V=96.0 pitch=-4.75 elev=0.180 thr=0.83
1000/23660 elapsed=109.4s relaxed=0 strict=0 best score=333.37 V=96.0 pitch=-4.00 elev=0.180 thr=0.83
1500/23660 elapsed=162.9s relaxed=0 strict=0 best score=325.72 V=96.0 pitch=-3.25 elev=0.180 thr=0.83
2000/23660 elapsed=218.9s relaxed=0 strict=0 best score=320.61 V=96.0 pitch=-2.75 elev=0.180 thr=0.83
2500/23660 elapsed=275.4s relaxed=0 strict=0 best score=312.91 V=96.0 pitch=-2.00 elev=0.180 thr=0.83
3000/23660 elapsed=330.8s relaxed=0 strict=0 best score=312.91 V=96.0 pitch=-2.00 elev=0.180 thr=0.83
3500/23660 elapsed=387.8s relaxed=0 strict=0 best score=312.91 V=96.0 pitch=-2.00 elev=0.180 thr=0.83
4000/23660 elapsed=445.4s relaxed=0 strict=0 best score=312.91 V=96.0 pitch=-2.00 elev=0.180 thr=0.83
4500/23660 elapsed=503.1s relaxed=0 

: 

## 3. Candidate Tables

In [ ]:
metric_cols = [
    'aircraft','init_alt_ft','init_speed_kts','init_pitch_deg','elevator','throttle','mixture',
    'score','feasible_relaxed','feasible_strict','abort',
    'eval_alt_drift_ft','eval_alt_rmse_ft','eval_tail_vs_fps','eval_speed_drift_kts',
    'eval_q_rms_deg_s','eval_tail_q_rms_deg_s','eval_theta_drift_deg',
    'eval_max_abs_pitch_deg','eval_max_abs_alpha_deg','eval_final_alt_ft','eval_final_speed_kts','eval_final_pitch_deg',
    'full_alt_drift_ft','full_tail_vs_fps','full_speed_drift_kts','full_q_rms_deg_s'
]
metric_cols = [c for c in metric_cols if c in refine_df.columns]

best_any = refine_df.iloc[0]
best_relaxed = refine_df[refine_df['feasible_relaxed'].fillna(False)].iloc[0] if refine_df['feasible_relaxed'].fillna(False).any() else best_any
best_strict = refine_df[refine_df['feasible_strict'].fillna(False)].iloc[0] if refine_df['feasible_strict'].fillna(False).any() else None

selected_rows = []
for label, row in [('best_score', best_any), ('best_relaxed', best_relaxed)]:
    d = row[metric_cols].to_dict()
    d['selection_label'] = label
    selected_rows.append(d)
if best_strict is not None:
    d = best_strict[metric_cols].to_dict()
    d['selection_label'] = 'best_strict'
    selected_rows.append(d)

selected_df = pd.DataFrame(selected_rows)
selected_path = os.path.join(SAVE_DIR, f'empirical_level_hold_selected_candidates_v49_{RUN_MODE}.csv')
selected_df.to_csv(selected_path, index=False)
print('Saved selected candidates:', selected_path)
display(selected_df)

print('Top by score:')
display(refine_df[metric_cols].head(TOP_K))

print('Top relaxed feasible only:')
display(refine_df[refine_df['feasible_relaxed'].fillna(False)][metric_cols].head(TOP_K))

## 4. Plot Selected Holds

In [ ]:
def rerun_from_row(row):
    _, log = run_case(row['init_speed_kts'], row['init_pitch_deg'], row['elevator'], row['throttle'], store_log=True)
    return log

plot_rows = []
for _, r in selected_df.iterrows():
    label = r['selection_label']
    log = rerun_from_row(r)
    if log is not None and len(log):
        log = log.copy()
        log['selection_label'] = label
        plot_rows.append(log)

if plot_rows:
    fig, axes = plt.subplots(5, 1, figsize=(12, 12), sharex=True)
    for log in plot_rows:
        label = log['selection_label'].iloc[0]
        t = log['time_s'].to_numpy(dtype=float)
        axes[0].plot(t, log['h_sl_ft'], label=label)
        axes[1].plot(t, log['vc_kts'], label=label)
        axes[2].plot(t, log['theta_deg'], label=label)
        axes[3].plot(t, log['q_deg_s'], label=label)
        axes[4].plot(t, log['elevator_cmd'], label=f'{label} elev')
        axes[4].plot(t, log['throttle_cmd'], '--', label=f'{label} thr')
    axes[0].axhline(INIT_ALT_FT, color='k', ls='--', lw=0.8)
    axes[0].set_ylabel('Altitude ft')
    axes[1].set_ylabel('Speed kt')
    axes[2].set_ylabel('Pitch deg')
    axes[3].set_ylabel('q deg/s')
    axes[4].set_ylabel('Controls')
    axes[4].set_xlabel('Time s')
    for ax in axes:
        ax.grid(True, alpha=0.3)
        ax.legend(loc='best')
    fig.suptitle('Refined empirical level-flight candidates')
    plt.tight_layout()
    fig_path = os.path.join(SAVE_DIR, f'empirical_level_hold_refined_candidates_v49_{RUN_MODE}.png')
    fig.savefig(fig_path, dpi=160)
    print('Saved:', fig_path)
    plt.show()
else:
    print('No logs available for plotting.')

## 5. Recommended Controller Initial Condition

For controller experiments, prefer `best_relaxed` if strict feasibility is unavailable. Use:

- `init_alt_ft`
- `init_speed_kts`
- `init_pitch_deg`
- initial `elevator`
- initial `throttle`
- `mixture`

Then let PID/MPPI take over after the same short settle period, and exclude the first few seconds from tracking metrics when comparing controllers.